In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [15]:
!rm -rf /kaggle/working/*

In [16]:
"""
SarcasmLens – Baseline System (Subtask 2)
-----------------------------------------
Goal:
1. Train ML models on Train/Validation data.
2. Evaluate and compare Accuracy, Precision, Recall, and F1 for ALL models.
3. Save the best model and evaluate it on the Test set.
"""

import pandas as pd
import numpy as np
import re
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# =======================================================
# 1. Configuration & Paths
# =======================================================
# UPDATED PATHS
TRAIN_PATH = "/kaggle/input/nlp-dataset/train.csv"
VAL_PATH = "/kaggle/input/nlp-dataset/validation.csv"
TEST_PATH = "/kaggle/input/nlp-dataset/test.csv"

# Output Directory (Working Directory)
OUTPUT_DIR = "/kaggle/working/"
METRICS_FILE = os.path.join(OUTPUT_DIR, "validation_metrics_summary.csv")
TEST_METRICS_FILE = os.path.join(OUTPUT_DIR, "test_metrics_summary.csv")

print("===== Loading Data =====")
try:
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    print(f"Train shape: {train_df.shape}")
    print(f"Validation shape: {val_df.shape}")
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    exit()

# Standardize column names to 'text' and 'label'
def standardize_cols(df):
    text_col = [c for c in df.columns if "tweet" in c.lower() or "text" in c.lower()][0]
    label_col = [c for c in df.columns if "label" in c.lower()][0]
    return df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})

train_df = standardize_cols(train_df)
val_df = standardize_cols(val_df)

# =======================================================
# 2. Preprocessing
# =======================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)       # URLs
    text = re.sub(r"@[A-Za-z0-9_]+", "", text)       # Mentions
    text = re.sub(r"#", "", text)                    # Hashtags
    text = re.sub(r"[^a-zA-Z\u0900-\u097F!?'\s]", " ", text) # English + Hindi + Punctuation
    text = re.sub(r"\s+", " ", text).strip()         
    return text

train_df["text"] = train_df["text"].apply(clean_text)
val_df["text"] = val_df["text"].apply(clean_text)

X_train, y_train = train_df["text"], train_df["label"]
X_val, y_val = val_df["text"], val_df["label"]

# =======================================================
# 3. Model Setup
# =======================================================
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "LogisticRegression": LogisticRegression(max_iter=1000, solver="liblinear", random_state=42),
    "LinearSVC": LinearSVC(C=1.0, random_state=42),
}

# =======================================================
# 4. Training Loop
# =======================================================
results_data = []
best_f1 = 0.0
best_model_name = None
best_pipeline = None

print("\n===== Starting Training & Evaluation =====")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Create Pipeline
    clf = Pipeline([
        ("tfidf", tfidf),
        ("model", model)
    ])
    
    # Fit on Train
    clf.fit(X_train, y_train)
    
    # Predict on Validation
    y_pred = clf.predict(X_val)
    
    # --- CALCULATE ALL METRICS (Weighted for Multiclass/Imbalance safety) ---
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted')
    rec = recall_score(y_val, y_pred, average='weighted')
    f1 = f1_score(y_val, y_pred, average='weighted')
    
    # Save Model
    model_filename = os.path.join(OUTPUT_DIR, f"{name}_model.pkl")
    joblib.dump(clf, model_filename)
    
    # Store Metrics
    results_data.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4)
    })
    
    # Track Best Model (based on F1)
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_pipeline = clf

# =======================================================
# 5. Validation Results Table
# =======================================================
metrics_df = pd.DataFrame(results_data)
metrics_df.sort_values(by="F1_Score", ascending=False, inplace=True)

# Save to CSV
metrics_df.to_csv(METRICS_FILE, index=False)

print("\n" + "="*60)
print("VALIDATION RESULTS (All Metrics)")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60 + "\n")
print(f"Best Model Selected: {best_model_name}")

# =======================================================
# 6. Test Set Evaluation (Best Model Only)
# =======================================================
print(f"\n===== Evaluating Best Model ({best_model_name}) on Test Set =====")

try:
    df_test = pd.read_csv(TEST_PATH)
    
    # Handle Test Columns
    possible_test_text = [col for col in df_test.columns if "tweet" in col.lower() or "text" in col.lower()]
    possible_test_label = [col for col in df_test.columns if "label" in col.lower()]
    
    text_col_test = possible_test_text[0] if possible_test_text else df_test.columns[1]
    label_col_test = possible_test_label[0] if possible_test_label else None

    # Rename
    if label_col_test:
        df_test = df_test[[text_col_test, label_col_test]].rename(columns={text_col_test: "text", label_col_test: "label"})
    else:
        # If no label, we can't calculate metrics
        df_test = df_test[[text_col_test]].rename(columns={text_col_test: "text"})
    
    # Clean
    df_test["text"] = df_test["text"].apply(clean_text)
    
    # Predict
    y_pred_test = best_pipeline.predict(df_test["text"])
    
    # --- TEST METRICS ---
    if "label" in df_test.columns:
        y_true_test = df_test["label"]
        
        t_acc = accuracy_score(y_true_test, y_pred_test)
        t_prec = precision_score(y_true_test, y_pred_test, average='weighted')
        t_rec = recall_score(y_true_test, y_pred_test, average='weighted')
        t_f1 = f1_score(y_true_test, y_pred_test, average='weighted')
        
        # Create DataFrame for Test Results
        test_res_df = pd.DataFrame([{
            "Model": best_model_name,
            "Accuracy": round(t_acc, 4),
            "Precision": round(t_prec, 4),
            "Recall": round(t_rec, 4),
            "F1_Score": round(t_f1, 4)
        }])
        
        test_res_df.to_csv(TEST_METRICS_FILE, index=False)
        
        print("\n" + "="*60)
        print("TEST SET RESULTS")
        print("="*60)
        print(test_res_df.to_string(index=False))
        print("="*60)
        print("\nDetailed Classification Report (Test Set):")
        print(classification_report(y_true_test, y_pred_test, digits=4))
        
    else:
        print("No label column found in Test set. Skipping metric calculation.")

    # Save Predictions
    df_test["predicted_label"] = y_pred_test
    pred_file = os.path.join(OUTPUT_DIR, "test_predictions.csv")
    df_test.to_csv(pred_file, index=False)
    print(f"\nPredictions saved to: {pred_file}")

except FileNotFoundError:
    print(f"Error: Test file not found at {TEST_PATH}")
except Exception as e:
    print(f"An error occurred during testing: {e}")

===== Loading Data =====
Train shape: (8456, 3)
Validation shape: (2114, 3)

===== Starting Training & Evaluation =====
Training RandomForest...
Training LogisticRegression...
Training LinearSVC...

VALIDATION RESULTS (All Metrics)
             Model  Accuracy  Precision  Recall  F1_Score
         LinearSVC    0.9674     0.9675  0.9674    0.9674
LogisticRegression    0.9622     0.9624  0.9622    0.9622
      RandomForest    0.9555     0.9586  0.9555    0.9559

Best Model Selected: LinearSVC

===== Evaluating Best Model (LinearSVC) on Test Set =====

TEST SET RESULTS
    Model  Accuracy  Precision  Recall  F1_Score
LinearSVC    0.9744     0.9746  0.9744    0.9744

Detailed Classification Report (Test Set):
              precision    recall  f1-score   support

          NO     0.9540    0.9703    0.9621       706
         YES     0.9849    0.9765    0.9807      1403

    accuracy                         0.9744      2109
   macro avg     0.9695    0.9734    0.9714      2109
weighted avg 